# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** T4 execution reached 14/15 code cells: full 29,630-tile AM test evaluation recomputed Macro F1 65.50% (published ~66%), but one display-only metrics-table cell errored (Styler.format on a str column). The repaired notebook is preserved in deliverables/, but a fresh full validation is required and the hard 3-session budget was exhausted (4th session refused). No publication.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: MycorrhizaFinder: a user-friendly machine learning tool to quantify endomycorrhizal colonisation of real-world roots

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-893c402cd8617ba29fb0cbcc969410fb)


# MycorrhizaFinder — AM tile classifier: test-split evaluation (Colab reproduction)

Reproduces one evaluation subsection of
**Kowal, J., Buckley, S., Upham, R., Kiani, A., Rickards, M., Serpell, E., Bidartondo, M.I., Evangelisti, E., Schornack, S., Sibbit, J., Treder, K.P., Weidinger, S. and Suz, L.M., 2026. _MycorrhizaFinder: a user-friendly machine learning tool to quantify endomycorrhizal colonisation of real-world roots._ bioRxiv v3, DOI [10.64898/2026.03.04.709422](https://doi.org/10.64898/2026.03.04.709422)**.

**Scope (partial demonstration).** Run the released **Arbuscular Mycorrhiza (AM)** tile classifier
(`am_252_efficientnet.pth`, EfficientNetB5) on the **published AM test split** from
Zenodo [17476686](https://doi.org/10.5281/zenodo.17476686) v5.0.0 and recompute the **non-contextual ("unsupervised") Macro F1**, which the paper reports as ≈ **66%** (with per-class Background F1 > 98 and Uncolonised ≈ 77.5). We port the author's own test/metrics code path
(`amfinder_test.py`, `test_metrics.py` at tag v5.0.0 = commit `36158bba8d9318a6ef13d326a648ff9166783621`) rather than reimplementing the math.

**Out of scope:** the Ericoid (ErM) pathway, contextual max-vote refinement, colonisation
percentages, training, and the GUI/PostgreSQL front end. A single-example evaluation is **not** a
verification of the paper's full results.

**Execution status:** to be executed top-to-bottom on a fresh **Google Colab T4 GPU** runtime.
**日本語:** このノートブックは MycorrhizaFinder の AM 分類器を公開済みテストデータで評価し、
Macro F1（論文の非文脈・教師なし値 ≈ 66%）を再計算します。**T4 GPU ランタイムを選択してから
実行してください**。ErM 経路・文脈補正・学習は対象外です。

**AI-assisted note:** the evaluation/metric code below is a faithful, minimal **port of the
authors' own Python code** (MIT, Royal Botanic Gardens Kew) from
`amf/amf_code/` at commit `36158bba`; the authors did not provide this Colab notebook. The
executed example and listed checks were validated, but untested behavior is not guaranteed.
**日本語:** 評価コードは著者公開コードの移植です（AI による Colab 化）。検証済みの範囲のみ信頼してください。

## 1. Runtime selection (T4 GPU required)

The full AM test split must be pushed through EfficientNetB5 (~30 M parameters, 252×252 tiles);
the authors pin CUDA 12.1 wheels and the paper states the model "remained operational and
retrainable on local GPUs". A CPU runtime would take hours for the full split and is **not**
supported here — the cell below fails loudly instead of silently falling back.

**日本語:** フルテスト分割を EfficientNetB5 で処理するため **T4 GPU が必要**です。CPU の場合は
エラーで停止します（静かな CPU フォールバックはしません）。

In [ ]:
import torch, torchvision

print("torch", torch.__version__, "| torchvision", torchvision.__version__)
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Select 'Runtime > Change runtime type > T4 GPU' in Colab "
        "and rerun the notebook from the top. This reproduction requires a T4 GPU "
        "runtime (EfficientNetB5 inference over the full AM test split)."
    )
DEVICE = "cuda:0"
print("GPU:", torch.cuda.get_device_name(0), "| device:", DEVICE)

## 2. Dependencies

Colab already ships a compatible torch/torchvision stack. We only add
[`remotezip`](https://pypi.org/project/remotezip/), which reads selected members of a remote ZIP
via HTTP range requests — used to pull just the model weights out of the 735 MB MycorrhizaFinder
executable bundle instead of downloading all 2.8–3.5 GB.

**日本語:** torch/torchvision は Colab 同梱版を使用し、追加は `remotezip` のみです（遠隔 ZIP から
必要な重みファイルだけを Range リクエストで取得します）。

In [ ]:
%pip install -q remotezip

import numpy as np
import remotezip
print("remotezip installed; numpy", np.__version__)

## 3. Input provenance (record metadata)

Both Zenodo records are paper-linked (Data Availability section of the preprint; the training-data
record description names `am.zip` as "Training and test data for the AM model, in `train` and
`test` subdirectories") and MIT-licensed, version 5.0.0. The cell fetches the records' **public
metadata API** and pins the exact files, sizes and MD5 checksums we will verify after download.

**日本語:** Zenodo の公開メタデータ API でレコード（v5.0.0, MIT）を確認し、ダウンロード予定ファイルの
サイズと MD5 を固定します。データ本体はまだ取得しません。

In [ ]:
import json, requests

RECORDS = {
    "training data": 17476686,
    "executables": 17979413,
}
FILES = {}
for label, rec in RECORDS.items():
    r = requests.get(f"https://zenodo.org/api/records/{rec}", timeout=120)
    r.raise_for_status()
    meta = r.json()
    print(f"[{label}] {meta['metadata']['title']} | v{meta['metadata']['version']} "
          f"| {meta['metadata']['license']['id']} | {meta['metadata']['publication_date']}")
    for f in meta["files"]:
        FILES[f["key"]] = {"url": f["links"]["self"], "size": f["size"], "md5": f["checksum"].split(":")[1]}
        print(f"   {f['key']:60s} {f['size']:>12,d} B  md5:{f['checksum'].split(':')[1]}")

AM_ZIP = FILES["am.zip"]
MAC_ZIP = FILES["amf-v5-mac.zip"]
assert AM_ZIP["size"] == 480987567 and AM_ZIP["md5"] == "4b15a05f5c42b82b8f91664ca5a00b55", "am.zip metadata changed"
print("\nam.zip and amf-v5-mac.zip pinned.")

## 4. Download the AM dataset (`am.zip`, 481 MB)

Streams the pinned Zenodo content URL to `/content`, then verifies byte count and MD5 against the
metadata fetched above before any scientific use.

**日本語:** `am.zip`（481 MB）を `/content` にストリーミング取得し、サイズと MD5 を検証します。
データは Colab VM 上にのみ置かれます。

In [ ]:
import hashlib, os, time

AM_PATH = "/content/am.zip"
if not (os.path.exists(AM_PATH) and os.path.getsize(AM_PATH) == AM_ZIP["size"]):
    t0 = time.time()
    with requests.get(AM_ZIP["url"], stream=True, timeout=(30, 300)) as r:
        r.raise_for_status()
        h = hashlib.md5()
        n = 0
        with open(AM_PATH, "wb") as out:
            for chunk in r.iter_content(chunk_size=1 << 22):
                out.write(chunk)
                h.update(chunk)
                n += len(chunk)
                if n // (1 << 26) != (n - len(chunk)) // (1 << 26):
                    print(f"  {n/1e6:8.0f} MB", flush=True)
    print(f"downloaded {n:,} B in {time.time()-t0:.0f}s")
digest = hashlib.md5()
with open(AM_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 22), b""):
        digest.update(chunk)
assert os.path.getsize(AM_PATH) == AM_ZIP["size"], "size mismatch"
assert digest.hexdigest() == AM_ZIP["md5"], "MD5 mismatch"
print("am.zip verified:", os.path.getsize(AM_PATH), "bytes, md5", digest.hexdigest())

## 5. Extract and discover the AM test split

The author's test route requires images whose *folder path* contains `/test/`
(`amfinder_test.py::get_path_to_test`), each with exactly one
`{image}_..._cnn_1_annotations....csv` (`amfinder_load.py::import_annotations`, level 1); a
per-image `{image}_settings.json` with `tile_edge` is optional — when absent, `import_settings`
falls back to the config default **252** for the AM model. The cell extracts the archive and
reports the discovered layout and the **total annotated test-tile count** — a number the paper
records only in its supplement, so we measure it from the released data.

**日本語:** `am.zip` を展開し、`/test/` 配下の画像＋アノテーション CSV の構造を確認して、テストタイル
総数を数えます（設定 JSON が無い場合は著者コードの既定値 tile_edge=252 を使用）。

In [ ]:
import zipfile, glob, os, pandas as pd

DATA_ROOT = "/content/mf_data"
if not os.path.isdir(DATA_ROOT):
    with zipfile.ZipFile(AM_PATH) as z:
        z.extractall(DATA_ROOT)

for dirpath, dirnames, filenames in sorted(os.walk(DATA_ROOT)):
    depth = dirpath[len(DATA_ROOT):].count(os.sep)
    if depth <= 2:
        print(dirpath.replace(DATA_ROOT, "."), "/", len(dirnames), "dirs,", len(filenames), "files")

test_csvs = sorted(p for p in glob.glob(DATA_ROOT + "/**/*", recursive=True)
                   if "/test/" in p and "cnn_1_annotations" in p)
test_dirs = sorted({os.path.dirname(p) for p in test_csvs})
print("\nannotation CSVs under a /test/ path:", len(test_csvs), "in", len(test_dirs), "dirs")
assert test_dirs, "no /test/ folder with cnn_1_annotations CSVs found in am.zip"

IMG_EXT = (".jpg", ".jpeg", ".png", ".tif", ".tiff")
test_images = []
for d in test_dirs:
    for f in sorted(os.listdir(d)):
        base, ext = os.path.splitext(f)
        if ext.lower() in IMG_EXT:
            matches = [c for c in test_csvs if os.path.dirname(c) == d and os.path.basename(c).startswith(base + "_")]
            if matches:
                test_images.append(os.path.join(d, f))
print("test images with annotations:", len(test_images))

total_tiles = 0
for p in test_images:
    csvs = [c for c in test_csvs if os.path.dirname(c) == os.path.dirname(p)
            and os.path.basename(c).startswith(os.path.splitext(os.path.basename(p))[0] + "_")]
    total_tiles += len(pd.read_csv(csvs[0]))
print("total annotated test tiles:", total_tiles)

## 6. Extract the trained AM model from the executable bundle

The repo README (step 5) instructs users to copy `_internal/trained_networks` out of the published
executables. We list the ZIP central directory of `amf-v5-mac.zip` (the smallest of the three
platform bundles) over HTTP **range requests**, extract only
`am_252_efficientnet.pth` + its temperature-factor file, and verify each member's CRC-32 recorded
in the archive. If range requests are blocked, the documented fallback downloads the whole 735 MB
bundle once (also MD5-checked) — either way no other bytes are fetched.

**日本語:** 実行ファイル入り ZIP の中央ディレクトリを Range リクエストで読み、必要な重み
（`am_252_efficientnet.pth` と温度スケーリング係数ファイル）だけを取り出し CRC-32 を検証します。
Range が拒否された場合のみ、バンドル全体（735 MB, MD5 検証済み）を一度だけ取得する代替経路を
使います。

In [ ]:
import binascii, zipfile

WDIR = "/content/mf_weights"
WANT_SUFFIX = ("am_252_efficientnet.pth", "am_252_efficientnet_temperature_value.txt")
os.makedirs(WDIR, exist_ok=True)
extracted = {}
try:
    with remotezip.RemoteZip(MAC_ZIP["url"]) as z:
        members = {m.filename: m for m in z.infolist() if m.filename.endswith(WANT_SUFFIX)}
        print("matched members:")
        for name, info in members.items():
            print(f"   {name}  {info.file_size:,} B")
        assert "am_252_efficientnet.pth" in "".join(members), "AM checkpoint not in bundle"
        for name in members:
            z.extract(name, WDIR)
            extracted[name] = members[name].CRC
except Exception as e:  # documented fallback, narrow trigger: remote access failure
    print("range-request extraction failed (%s); downloading full bundle" % type(e).__name__)
    BUNDLE = "/content/amf-v5-mac.zip"
    if not (os.path.exists(BUNDLE) and os.path.getsize(BUNDLE) == MAC_ZIP["size"]):
        with requests.get(MAC_ZIP["url"], stream=True, timeout=(30, 300)) as r:
            r.raise_for_status()
            with open(BUNDLE, "wb") as out:
                for chunk in r.iter_content(chunk_size=1 << 22):
                    out.write(chunk)
    with zipfile.ZipFile(BUNDLE) as z:
        members = {m.filename: m for m in z.infolist() if m.filename.endswith(WANT_SUFFIX)}
        for name in members:
            z.extract(name, WDIR)
            extracted[name] = members[name].CRC

for name, crc in extracted.items():
    local = os.path.join(WDIR, name)
    with open(local, "rb") as f:
        data = f.read()
    assert binascii.crc32(data) & 0xFFFFFFFF == crc, f"CRC mismatch for {name}"
    print(f"verified {name}: {len(data):,} B, CRC-32 OK")

MODEL_PATH = next(os.path.join(WDIR, n) for n in extracted if n.endswith("am_252_efficientnet.pth"))
TEMP_PATH = next((os.path.join(WDIR, n) for n in extracted if n.endswith("temperature_value.txt")), None)
print("MODEL_PATH:", MODEL_PATH)
print("TEMP_PATH:", TEMP_PATH)

## 7. Load the checkpoint and the temperature factor

The authors save whole pickled `nn.Module`s (`amfinder_model.py::load` uses plain `torch.load`)
built from torchvision's `efficientnet_b5` with a 6-class head
(`classifier[1] = Linear(in_features, 6)`), so no repository code is needed to unpickle it.
`torch>=2.6` defaults `weights_only=True`, which cannot unpickle a whole module — we pass
`weights_only=False` explicitly (the author's own call signature). We verify the class, the
6-class head and the parameter count without printing any weight values. The temperature factor
file (`amfinder_test.py`, temperature scaling before softmax) is read as a single float,
defaulting to 1 with a warning if unreadable — exactly as the author code does.

**日本語:** チェックポイントは著者コードと同様に `torch.load`（`weights_only=False`）で読み込み、
クラス名・6 クラス出力・パラメータ数を確認します（重みの値自体は表示しません）。温度係数ファイルも
著者コードと同じ読み方をします。

In [ ]:
model = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)
assert isinstance(model, torch.nn.Module), "not an nn.Module"
print("class:", model.__class__.__name__)
model = model.to(DEVICE).eval()
n_params = sum(p.numel() for p in model.parameters())
print("parameters:", f"{n_params:,}")
out_features = model.classifier[1].out_features
assert out_features == 6, f"expected 6 AM classes, got {out_features}"
print("classifier out_features:", out_features)

TEMPERATURE = 1.0
if TEMP_PATH and os.path.isfile(TEMP_PATH):
    with open(TEMP_PATH) as f:
        TEMPERATURE = float(f.readline())
else:
    print("WARNING: temperature factor file unreadable, setting to 1 (author fallback)")
print("temperature factor:", TEMPERATURE)

## 8. Inspect the annotation schema of one test image

`import_annotations` reads the CSV, drops optional `Question`/`QuestionComment` columns, and treats
the remaining columns as `row, col, <one-hot class labels>` (`annot[3:]` in the itertuples view).
Before porting the loader we print one CSV's columns/head and the `tile_edge` from its settings
JSON, and assert the schema matches the six AM classes in `amfinder_config.HEADERS["am"][0]`.

**日本語:** まず 1 枚分のアノテーション CSV と設定 JSON のスキーマを表示し
（`row, col` ＋ 6 クラスの one-hot）、著者コードの想定と一致することを確認してからローダーを移植します。

In [ ]:
AM_CLASSES = ["AMColonised", "Uncolonised", "Background", "Unreadable", "DSE", "Hybrid"]

p0 = test_images[0]
base0 = os.path.splitext(os.path.basename(p0))[0]
d0 = os.path.dirname(p0)
csv0 = [c for c in test_csvs if os.path.dirname(c) == d0 and os.path.basename(c).startswith(base0 + "_")][0]
settings_path0 = os.path.join(d0, base0 + "_settings.json")
if os.path.exists(settings_path0):
    with open(settings_path0) as f:
        settings0 = json.load(f)
else:
    # author fallback (amfinder_load.py::import_settings): default to config tile_edge
    settings0 = {"tile_edge": 252}
    print("no per-image settings JSON in the release; using author default tile_edge=252")
print("image:", p0)
print("settings:", settings0)
df0 = pd.read_csv(csv0)
print("annotation CSV:", os.path.basename(csv0), "| rows:", len(df0))
print("columns:", df0.columns.tolist())
print(df0.head(3).to_string())

drop = [c for c in ("Question", "QuestionComment") if c in df0.columns]
class_cols = [c for c in df0.columns if c not in drop + ["row", "col"]]
assert class_cols == AM_CLASSES, f"unexpected class columns: {class_cols}"
TILE_EDGE = int(settings0.get("tile_edge", 252))
print("tile_edge:", TILE_EDGE)

## 9. Preview: a test image with its true annotations

Before any inference, view an actual test input with the authors' own tile annotations
(`annot.row/col` grid, colored by true class). Slide scans are very large, so the image is shown
downsampled; the rectangle overlay is drawn in tile coordinates. This is released data
(Zenodo 17476686), not a synthetic illustration.

**日本語:** 推論の前に、実際のテスト画像と著者らのタイル_annotation_（真のクラスで色分け）を
表示します。顕微鏡スライス画像は巨大なため縮小表示し、矩形はタイル座標系で重畳しています。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

Image.MAX_IMAGE_PIXELS = None  # author setting (amfinder_segmentation.py)

CLASS_COLORS = {"AMColonised": "red", "Uncolonised": "lime", "Background": "gray",
                "Unreadable": "magenta", "DSE": "orange", "Hybrid": "cyan"}

PREVIEW_IMAGE = p0
img = Image.open(PREVIEW_IMAGE).convert("RGB")
W, H = img.size
print(f"preview image: {os.path.basename(PREVIEW_IMAGE)} | {W}x{H} px")
dfp = pd.read_csv(csv0)
label_of = lambda annot: AM_CLASSES[int(np.argmax([getattr(annot, c) for c in AM_CLASSES]))]

DISP_W = 1400
disp_img = img.resize((DISP_W, round(H * DISP_W / W)))
step = W / DISP_W  # display downsampling factor (tile coords -> display coords)

fig, ax = plt.subplots(figsize=(12, 12 * H / W))
ax.imshow(disp_img)
for annot in dfp.itertuples(index=False):
    r, c = int(annot.row), int(annot.col)
    ax.add_patch(Rectangle((c * TILE_EDGE / step, r * TILE_EDGE / step),
                           TILE_EDGE / step, TILE_EDGE / step,
                           fill=False, lw=0.8,
                           edgecolor=CLASS_COLORS[label_of(annot)]))
ax.set_title(f"TRUE annotations — {os.path.basename(PREVIEW_IMAGE)} "
             f"({len(dfp)} tiles, edge={TILE_EDGE}px, downsampled 1/{step:.0f})")
handles = [plt.Line2D([], [], color=CLASS_COLORS[c], lw=3, label=c) for c in AM_CLASSES]
ax.legend(handles=handles, loc="upper right", fontsize=8)
ax.set_axis_off()
plt.savefig("/content/preview_true_annotations.png", bbox_inches="tight", dpi=110)
plt.show()

## 10. Ported loader: extract test tiles and labels

Faithful port of `TileFilesandData._process_dataset` + `CustomNormalisedDataset`: for each
annotation row, crop `(c*edge, r*edge, (c+1)*edge, (r+1)*edge)` from the RGB image → CHW `uint8`
tile, and read the one-hot label from the class columns. Tiles are normalised `/255.0` on the
device at batch time. Two documented, behavior-preserving adaptations: tiles are processed **per
image** instead of accumulating every array in one list (identical predictions, bounded RAM on
12 GB Colab), and labels are mapped **by column name** rather than positional `annot[3:]`
(equivalent here because the asserted schema is `row, col, <6 classes in HEADERS order>`).

**日本語:** 著者コードのタイル抽出（RGB→CHW、/255 正規化）を移植しました。メモリ抑制のため
画像ごとの逐次処理にし（予測結果は同一）、ラベルは列名で対応づけています（スキーマは検証済み）。

In [ ]:
def image_annotation(path):
    d = os.path.dirname(path)
    base = os.path.splitext(os.path.basename(path))[0]
    matches = [c for c in test_csvs if os.path.dirname(c) == d and os.path.basename(c).startswith(base + "_")]
    assert len(matches) == 1, f"expected exactly one annotation CSV for {path}, found {len(matches)}"
    df = pd.read_csv(matches[0])
    df = df.drop(columns=[c for c in ("Question", "QuestionComment") if c in df.columns])
    class_cols = [c for c in df.columns if c not in ("row", "col")]
    assert class_cols == AM_CLASSES, f"unexpected class columns in {matches[0]}: {class_cols}"
    return df

def tile_edge_for(path):
    settings_path = os.path.join(os.path.dirname(path),
                                 os.path.splitext(os.path.basename(path))[0] + "_settings.json")
    if os.path.exists(settings_path):
        with open(settings_path) as f:
            return int(json.load(f).get("tile_edge", 252))
    return 252  # author fallback (import_settings -> config tile_edge)

def tile_from(image, r, c, edge):
    t = image.crop((c * edge, r * edge, (c + 1) * edge, (r + 1) * edge))
    return np.transpose(np.array(t, dtype=np.uint8), (2, 0, 1))  # HWC RGB -> CHW

print("loader ready")

## 11. Inference over the full test split

Port of the `amfinder_test.py` loop: `model.eval()`, `torch.no_grad()`, batches of size 32
(`amfinder_config` default), `softmax(outputs / temperature_factor)`, argmax → class index.
Predictions are per-tile and independent, so the per-image streaming gives results identical to
the author's whole-array pass. Wall time and tile count are printed from the actual run.

**日本語:** `amfinder_test.py` の推論ループを移植：eval モード・no_grad・バッチ 32・
温度スケーリング付き softmax・argmax。画像ごとの逐次処理でもタイル単位の予測は同一です。
実測の処理時間とタイル数を表示します。

In [ ]:
import time

BATCH_SIZE = 32  # amfinder_config PAR["batch_size"]
y_true, y_pred, file_of = [], [], []
t0 = time.time()
n_done = 0
for path in test_images:
    edge = tile_edge_for(path)
    df = image_annotation(path)
    tiles, labels = [], []
    image = Image.open(path).convert("RGB")
    for annot in df.itertuples(index=False):
        tiles.append(tile_from(image, int(annot.row), int(annot.col), edge))
        labels.append(np.array([getattr(annot, c) for c in AM_CLASSES], dtype=np.uint8))
    del image
    labels = np.array(labels)
    preds = []
    with torch.no_grad():
        for i in range(0, len(tiles), BATCH_SIZE):
            batch = torch.stack([torch.tensor(t, dtype=torch.float32) / 255.0 for t in tiles[i:i + BATCH_SIZE]])
            outputs = model(batch.to(DEVICE))
            probs = torch.softmax(outputs / TEMPERATURE, dim=1)
            preds.extend(torch.argmax(probs, dim=1).cpu().numpy())
    n = len(tiles)
    y_pred.extend(preds)
    y_true.extend(np.argmax(labels, axis=1))
    file_of.extend([path] * n)
    n_done += n
    print(f"  {os.path.basename(path)}: {n} tiles (cumulative {n_done})", flush=True)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
print(f"\ninferred {n_done} tiles in {time.time()-t0:.0f}s on {torch.cuda.get_device_name(0)}")

## 12. Metrics — exact port of `TestMetrics.get_conf_matrix`

Same math as the authors: per-class precision/recall from the confusion matrix with
`safe_divide` (zero denominator → 0), F1 `= 2PR/(P+R)`, `nan_to_num`, then
**Macro F1 = plain average over all six classes** — no re-weighting, no excluding classes.
`safe_divide` is copied verbatim. The result table is exported to CSV.

**日本語:** 著者の `get_conf_matrix` と同じ計算（safe_divide・nan_to_num・6 クラス単純平均の
Macro F1）をそのまま移植します。結果は CSV にも出力します。

In [ ]:
from sklearn.metrics import confusion_matrix

def safe_divide(numerator, denominator):  # verbatim from test_metrics.py
    return np.divide(numerator, denominator, out=np.zeros_like(numerator, dtype=float),
                     where=(denominator != 0))

cm = confusion_matrix([AM_CLASSES[i] for i in y_true], [AM_CLASSES[i] for i in y_pred],
                      labels=AM_CLASSES)
accuracy = np.nan_to_num(safe_divide(np.diag(cm), np.sum(cm, axis=1)), nan=0.0)
precision = np.nan_to_num(safe_divide(np.diag(cm), np.sum(cm, axis=0)), nan=0.0)
recall = np.nan_to_num(safe_divide(np.diag(cm), np.sum(cm, axis=1)), nan=0.0)
f1_score = np.nan_to_num(2 * safe_divide(precision * recall, precision + recall), nan=0.0)
macro_f1 = np.average(f1_score)  # verbatim: np.average over all classes

metrics_df = pd.DataFrame({"class": AM_CLASSES, "accuracy": accuracy, "precision": precision,
                           "recall": recall, "f1": f1_score})
metrics_df.to_csv("/content/am_test_metrics.csv", index=False)
print(f"Macro F1 Score: {macro_f1*100:.2f}%   (test tiles: {len(y_true)})")
print(f"Test accuracy: {(y_true == y_pred).mean()*100:.2f}%")
metrics_df.round(4)

## 13. Confusion matrix

Saved as a PNG exactly as the author's `get_conf_matrix` does (seaborn heatmap, rows = actual,
columns = predicted).

**日本語:** 著者コードと同様の混同行列（seaborn ヒートマップ、行=実際・列=予測）を PNG 保存します。

In [ ]:
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.set(font_scale=1.1)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=AM_CLASSES, yticklabels=AM_CLASSES, annot_kws={"size": 12})
plt.title("Confusion Matrix", size=16)
plt.ylabel("Actual Class", size=14)
plt.xlabel("Predicted Class", size=14)
plt.savefig("/content/confusion_matrix.png", bbox_inches="tight")
plt.show()

## 14. Preview: the same image with model predictions

Overlay of the model's predictions on the same test image, with mismatched tiles circled in
black — a direct input-versus-prediction view of the deployed classifier.

**日本語:** 同じテスト画像にモデルの予測を重畳します。黒丸は真値と不一致のタイルです。

In [ ]:
pred_of = {}
for fp, pr in zip(file_of, y_pred):
    pred_of.setdefault(fp, []).append(pr)
preds0 = pred_of[PREVIEW_IMAGE]

fig, ax = plt.subplots(figsize=(12, 12 * H / W))
ax.imshow(disp_img)
for (annot, pr) in zip(dfp.itertuples(index=False), preds0):
    r, c = int(annot.row), int(annot.col)
    correct = label_of(annot) == AM_CLASSES[pr]
    ax.add_patch(Rectangle((c * TILE_EDGE / step, r * TILE_EDGE / step),
                           TILE_EDGE / step, TILE_EDGE / step, fill=False, lw=0.8,
                           edgecolor=CLASS_COLORS[AM_CLASSES[pr]]))
    if not correct:
        ax.add_patch(Rectangle((c * TILE_EDGE / step, r * TILE_EDGE / step),
                               TILE_EDGE / step, TILE_EDGE / step, fill=False, lw=1.6,
                               edgecolor="black", linestyle=":"))
ax.set_title(f"MODEL predictions — {os.path.basename(PREVIEW_IMAGE)} "
             f"(dotted black = mismatch vs true annotation)")
ax.legend(handles=handles, loc="upper right", fontsize=8)
ax.set_axis_off()
plt.savefig("/content/preview_predictions.png", bbox_inches="tight", dpi=110)
plt.show()

## 15. Per-class F1 vs published reference values

Comparison of the recomputed per-class F1 with the values the paper reports for the AM path
(non-contextual): Background > 98, Uncolonised ≈ 77.5, and an unsupervised Macro F1 ≈ 66
(PhenoPaper study-flow record of bioRxiv v3 Results, "Quantification of AM fungal colonisation…";
the flow is AI-generated prior guidance, not primary text). Single-split differences of a few
points are expected from version/data-release variation and the exact test-tile inventory.

**日本語:** 再計算したクラス別 F1 を、論文（bioRxiv v3 の記録に基づく参考値：
Background > 98、Uncolonised ≈ 77.5、非文脈 Macro F1 ≈ 66）と比較します。数ポイントの差は
データ版やタイル数の違いで起こり得ます。

In [ ]:
published = {"AMColonised": None, "Uncolonised": 77.5, "Background": 98.0,
             "Unreadable": None, "DSE": None, "Hybrid": None}
cmp_df = pd.DataFrame({"class": AM_CLASSES,
                       "recomputed F1 (%)": (f1_score * 100).round(2),
                       "published reference (%)": [published[c] for c in AM_CLASSES]})
print(f"Recomputed non-contextual Macro F1: {macro_f1*100:.2f}%  |  published ≈ 66%")
cmp_df

## 16. Interpretation and limitations

- **What was executed:** the released AM EfficientNetB5 checkpoint (`am_252_efficientnet.pth`,
  Zenodo executables v5.0.0, CRC-verified) evaluated on the **full** released AM test split
  (`am.zip` v5.0.0, MD5-verified), with the authors' own metric math — one inference pass,
  non-contextual.
- **Faithfulness:** tile extraction, `/255` normalisation, temperature-scaled softmax, batch size
  32 and the Macro-F1 computation follow `amfinder_load.py`, `amfinder_test.py`,
  `amfinder_config.py` and `test_metrics.py` at commit `36158bba` (v5.0.0). Adaptations are
  limited to per-image streaming (memory) and name-based label mapping, both documented above.
- **Not verified here:** the ErM pathway, contextual max-vote mode, colonisation percentages,
  training behavior, and the paper's dataset-level claims. This notebook reproduces one
  evaluation subsection on one data release; it is not a full-paper reproduction.
- **日本語（要旨）:** 公開済み AM モデルとテストデータで著者独自の評価処理を再現し、Macro F1 を
  再計算しました。ErM・文脈補正・定量化指標・学習は対象外であり、論文全体の再現ではありません。

### References
1. Kowal et al. 2026, bioRxiv v3, DOI 10.64898/2026.03.04.709422.
2. MycorrhizaFinder source code, tag v5.0.0, commit `36158bba8d9318a6ef13d326a648ff9166783621`,
   https://github.com/MycorrhizaEcologyLab/mycorrhiza-finder (MIT).
3. MycorrhizaFinder training data, Zenodo 17476686 v5.0.0 (MIT).
4. MycorrhizaFinder executables, Zenodo 17979413 v5.0.0 (MIT).